# Notebook to parse entries from the FungAMR database

In [1]:
import pandas as pd
import numpy as np

In [2]:
wt_dict = {
    'Fks1-HS1':(639,'FLVLSLRDP'),
    'Fks1-HS2':(1354,'DWVRRYTL'),
    'Fks1-HS3':(690,'LDTYLWYIIVN'),
    'Fks2-HS1':(658,'FLILSLRDP'),
    'Fks2-HS2':(1372,'DWVRRYTL'),
    'Fks2-HS3':(709,'LDTYLWYIVVN')
}

In [3]:
df = pd.read_csv('Fks_mutated_hotspot_seq_310325.csv', index_col=0)
df

,species,ortho_homolog,uniprot,mutation,aa,positions,Hotspot,gene or protein name,drug,pubmedid,confidence score,ortho_mut,MIC,Strongest resistance evidence reported,Strongest sensitivity evidence reported,Scer_mutation
0,Aspergillus fumigatus,Fks,Q4WLT4_ASPFU,L678Y,FLTLSFKDP,"[675, 676, 677, 678, 679, 680, 681, 682, 683]",HS1,Fks1,Caspofungin,16110824,1.0,NaN,4 µg/mL,1.0,NaN,L642Y
1,Candida albicans,Fks,A0A1D8PCT0_CANAL,D648Y,FLTLSLRYP,"[641, 642, 643, 644, 645, 646, 647, 648, 649]",HS1,Fks1,Anidulafungin,18955538,8.0,OM_4086,0.83 µg/mL,8.0,NaN,D646Y
2,Candida albicans,Fks,A0A1D8PCT0_CANAL,D648Y,FLTLSLRYP,"[641, 642, 643, 644, 645, 646, 647, 648, 649]",HS1,Fks1,Caspofungin,18955538,8.0,OM_4086,2.67 µg/mL,8.0,NaN,D646Y
3,Candida albicans,Fks,A0A1D8PCT0_CANAL,D648Y,FLTLSLRYP,"[641, 642, 643, 644, 645, 646, 647, 648, 649]",HS1,Fks1,Micafungin,18955538,8.0,OM_4086,0.83 µg/mL,8.0,NaN,D646Y
4,Candida albicans,Fks,A0A1D8PCT0_CANAL,F641C,CLTLSLRDP,"[641, 642, 643, 644, 645, 646, 647, 648, 649]",HS1,Fks1,Caspofungin,37746235,4.0,OM_3925,2.5 µg/mL,4.0,NaN,F639C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1244,Nakaseomyces glabratus,Fks,Q6FMZ3_CANGA,S663Y,FLILYLRDP,"[659, 660, 661, 662, 663, 664, 665, 666, 667]",HS1,Fks2,Caspofungin,30895218,8.0,OM_4026,0.5 µg/mL,8.0,NaN,S662Y
1245,Nakaseomyces glabratus,Fks,Q6FMZ3_CANGA,S663Y,FLILYLRDP,"[659, 660, 661, 662, 663, 664, 665, 666, 667]",HS1,Fks2,Micafungin,30895218,8.0,OM_4026,0.5 µg/mL,4.0,NaN,S662Y
1246,Nakaseomyces glabratus,Fks,Q6FMZ3_CANGA,W715L,LDTYLLYIVVN,"[710, 711, 712, 713, 714, 715, 716, 717, 718, ...",HS3,Fks2,Scy-078,28630180,4.0,OM_4128,4 µg/mL,4.0,NaN,W714L
1247,Nakaseomyces glabratus,Fks,Q6FMZ3_CANGA,W715L,LDTYLLYIVVN,"[710, 711, 712, 713, 714, 715, 716, 717, 718, ...",HS3,Fks2,Anidulafungin,29228268,4.0,OM_4128,2 µg/mL,4.0,NaN,W714L


In [4]:
df['locus'] = df['gene or protein name'] + '-' + df['Hotspot']

In [5]:
# Drop deletion mutants
df.drop(df[df.mutation.str.contains('del', regex=True)].index, inplace=True)

In [6]:
def mut_to_Scer_HS(mut, locus, hs_dict):
    if type(mut) != str:
        return np.nan
    
    alt_res = mut[-1]
    Scer_pos = int(mut[1:-1])
    
    pos0 = Scer_pos - hs_dict[locus][0]
    wtseq = hs_dict[locus][1]
    
    s = wtseq[:pos0] + alt_res + wtseq[pos0 + 1:]
    
    return s

In [7]:
mut_to_Scer_HS('L642Y', 'Fks1-HS1', wt_dict)

'FLVYSLRDP'

In [8]:
df['Scer_HS_mut'] = df.apply(lambda row: mut_to_Scer_HS(row.Scer_mutation, row.locus, wt_dict), axis=1)
df

,species,ortho_homolog,uniprot,mutation,aa,positions,Hotspot,gene or protein name,drug,pubmedid,confidence score,ortho_mut,MIC,Strongest resistance evidence reported,Strongest sensitivity evidence reported,Scer_mutation,locus,Scer_HS_mut
0,Aspergillus fumigatus,Fks,Q4WLT4_ASPFU,L678Y,FLTLSFKDP,"[675, 676, 677, 678, 679, 680, 681, 682, 683]",HS1,Fks1,Caspofungin,16110824,1.0,NaN,4 µg/mL,1.0,NaN,L642Y,Fks1-HS1,FLVYSLRDP
1,Candida albicans,Fks,A0A1D8PCT0_CANAL,D648Y,FLTLSLRYP,"[641, 642, 643, 644, 645, 646, 647, 648, 649]",HS1,Fks1,Anidulafungin,18955538,8.0,OM_4086,0.83 µg/mL,8.0,NaN,D646Y,Fks1-HS1,FLVLSLRYP
2,Candida albicans,Fks,A0A1D8PCT0_CANAL,D648Y,FLTLSLRYP,"[641, 642, 643, 644, 645, 646, 647, 648, 649]",HS1,Fks1,Caspofungin,18955538,8.0,OM_4086,2.67 µg/mL,8.0,NaN,D646Y,Fks1-HS1,FLVLSLRYP
3,Candida albicans,Fks,A0A1D8PCT0_CANAL,D648Y,FLTLSLRYP,"[641, 642, 643, 644, 645, 646, 647, 648, 649]",HS1,Fks1,Micafungin,18955538,8.0,OM_4086,0.83 µg/mL,8.0,NaN,D646Y,Fks1-HS1,FLVLSLRYP
4,Candida albicans,Fks,A0A1D8PCT0_CANAL,F641C,CLTLSLRDP,"[641, 642, 643, 644, 645, 646, 647, 648, 649]",HS1,Fks1,Caspofungin,37746235,4.0,OM_3925,2.5 µg/mL,4.0,NaN,F639C,Fks1-HS1,CLVLSLRDP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1244,Nakaseomyces glabratus,Fks,Q6FMZ3_CANGA,S663Y,FLILYLRDP,"[659, 660, 661, 662, 663, 664, 665, 666, 667]",HS1,Fks2,Caspofungin,30895218,8.0,OM_4026,0.5 µg/mL,8.0,NaN,S662Y,Fks2-HS1,FLILYLRDP
1245,Nakaseomyces glabratus,Fks,Q6FMZ3_CANGA,S663Y,FLILYLRDP,"[659, 660, 661, 662, 663, 664, 665, 666, 667]",HS1,Fks2,Micafungin,30895218,8.0,OM_4026,0.5 µg/mL,4.0,NaN,S662Y,Fks2-HS1,FLILYLRDP
1246,Nakaseomyces glabratus,Fks,Q6FMZ3_CANGA,W715L,LDTYLLYIVVN,"[710, 711, 712, 713, 714, 715, 716, 717, 718, ...",HS3,Fks2,Scy-078,28630180,4.0,OM_4128,4 µg/mL,4.0,NaN,W714L,Fks2-HS3,LDTYLLYIVVN
1247,Nakaseomyces glabratus,Fks,Q6FMZ3_CANGA,W715L,LDTYLLYIVVN,"[710, 711, 712, 713, 714, 715, 716, 717, 718, ...",HS3,Fks2,Anidulafungin,29228268,4.0,OM_4128,2 µg/mL,4.0,NaN,W714L,Fks2-HS3,LDTYLLYIVVN


In [9]:
gby = df.groupby(['locus','Scer_mutation','Scer_HS_mut','drug'])[['pubmedid','species','aa', 'mutation',
                                                                  'Strongest resistance evidence reported',
                                                                  'Strongest sensitivity evidence reported'
                                                                 ]].agg(pubmed_ids = ('pubmedid', 'unique'),
                                                                        species = ('species', 'unique'),
                                                                        mutations = ('mutation', 'unique'),
                                                                        aa = ('aa', 'unique'),
                                                                        best_res = ('Strongest resistance evidence reported', 'min'),
                                                                        best_sens = ('Strongest sensitivity evidence reported', 'max')
                                                                       ).reset_index()
gby

,locus,Scer_mutation,Scer_HS_mut,drug,pubmed_ids,species,mutations,aa,best_res,best_sens
0,Fks1-HS1,D646E,FLVLSLREP,Anidulafungin,"[18378714, 19546367, 23487382, 24829248]",[Nakaseomyces glabratus],[D632E],[FLILSLREP],7.0,NaN
1,Fks1-HS1,D646E,FLVLSLREP,Caspofungin,"[18378714, 19546367, 23487382, 24153129, 24829...",[Nakaseomyces glabratus],[D632E],[FLILSLREP],7.0,NaN
2,Fks1-HS1,D646E,FLVLSLREP,Echinocandin,[27020939],[Nakaseomyces glabratus],[D632E],[FLILSLREP],8.0,NaN
3,Fks1-HS1,D646E,FLVLSLREP,Fluconazole,"[23487382, 27020939]",[Nakaseomyces glabratus],[D632E],[FLILSLREP],8.0,NaN
4,Fks1-HS1,D646E,FLVLSLREP,Micafungin,"[18378714, 19546367, 23487382, 24153129, 24829...",[Nakaseomyces glabratus],[D632E],[FLILSLREP],7.0,NaN
...,...,...,...,...,...,...,...,...,...,...
312,Fks2-HS1,S662Y,FLILYLRDP,Fluconazole,[24126582],[Nakaseomyces glabratus],[S663Y],[FLILYLRDP],NaN,-8.0
313,Fks2-HS1,S662Y,FLILYLRDP,Micafungin,"[24126582, 24153129, 27872063, 30895218]",[Nakaseomyces glabratus],[S663Y],[FLILYLRDP],4.0,NaN
314,Fks2-HS3,W714L,LDTYLLYIVVN,Anidulafungin,[29228268],[Nakaseomyces glabratus],[W715L],[LDTYLLYIVVN],4.0,NaN
315,Fks2-HS3,W714L,LDTYLLYIVVN,Micafungin,[29228268],[Nakaseomyces glabratus],[W715L],[LDTYLLYIVVN],4.0,NaN


In [10]:
def annotate_phenotype(res_score, sens_score):
    if np.isnan(res_score):
        return 'sensitive'
    elif np.isnan(sens_score):
        return 'resistant'
    else:
        if res_score < -1*sens_score:
            return 'resistant'
        else:
            return 'sensitive'

In [11]:
gby['phenotype'] = gby.apply(lambda row: annotate_phenotype(row.best_res, row.best_sens), axis=1)

In [14]:
gby[gby.drug.isin(['Caspofungin','Micafungin','Anidulafungin','Echinocandin'])].reset_index(drop=True).to_csv('fungamrmut_df.csv', index=None)